# ⬡ Lab 04 — RAG Pipeline: Chunking, Retrieval & Evaluation
**End-to-end retrieval-augmented generation with production-grade strategies**

---
**Real-world scenario:** ING Bank has 10,000+ pages of regulatory documents, mortgage policies,
and product guides. Compliance officers spend hours searching for specific clauses.
You will build the RAG system that answers their natural-language questions — with full evaluation.

**What you will build:**
1. Compare three chunking strategies on the same document
2. Build a ChromaDB vector store and measure retrieval quality (Hit Rate, MRR)
3. Implement hybrid search (dense + BM25) with Reciprocal Rank Fusion
4. Add cross-encoder re-ranking
5. Build a full RAG pipeline and evaluate with RAGAS
6. Track all experiments with MLflow

**Estimated time:** 70 min | **Level:** Intermediate | **MLflow: built-in on Databricks**

In [ ]:
%pip install -q sentence-transformers chromadb rank-bm25 openai ragas datasets mlflow pandas numpy

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import mlflow
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from openai import OpenAI

embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
client_oai = OpenAI()

# MLflow experiment — on Databricks this appears in your Workspace
mlflow.set_experiment('ING-RAG-Pipeline')
print('Setup complete. MLflow tracking: active')

In [ ]:
# Synthetic ING policy document (representative content)
ING_MORTGAGE_POLICY = '''
ING MORTGAGE LENDING POLICY 2025

1. INCOME REQUIREMENTS
The maximum mortgage amount is calculated as 4.5 times the gross annual income of the primary applicant.
For joint applications, both incomes are included: 100% of the higher income plus 70% of the lower income.
Self-employed applicants must provide three years of annual accounts. The average of the three years is used,
unless the most recent year is lower, in which case that figure is used.

2. LOAN-TO-VALUE (LTV) REQUIREMENTS
For primary residences, the maximum LTV is 100% of the market value as assessed by a certified appraiser.
For second properties (investment or holiday), the maximum LTV is 90%.
Properties requiring significant renovation may be assessed against future value, subject to a building depot.
The National Mortgage Guarantee (NHG) is available for mortgages up to EUR 435,000 from 2025.

3. AFFORDABILITY STRESS TEST
ING applies a standard stress test interest rate of 5.0% over a 30-year term to assess affordability.
Total monthly obligations (mortgage plus existing loans) may not exceed 35% of gross monthly income.
Car leases, student loan repayments, and personal loans are included in the obligations calculation.
Childcare costs are not included in the affordability calculation per DNB guidelines.

4. ELIGIBLE PROPERTY TYPES
ING finances owner-occupied residential properties in the Netherlands and Belgium.
Commercial properties, agricultural land, and properties abroad are not eligible for residential mortgages.
Leasehold (erfpacht) properties are eligible if the remaining lease term exceeds the mortgage term by 10 years.
New construction properties can be financed with a construction depot.

5. REQUIRED DOCUMENTATION
Applicants must provide: a recent payslip (within 3 months), employer statement (werkgeversverklaring),
the most recent annual income statement (jaaropgave), a property valuation report, and valid identification.
Additional documentation may be required for non-EU income, bonus income, or investment income.

6. INTEREST RATE OPTIONS
Fixed rate periods available: 1, 5, 10, 15, 20, 25, and 30 years.
Variable rate mortgages are available and track the ECB base rate with a margin.
Rate locks can be requested up to 3 months before purchase completion.
Penalty-free overpayment of up to 10% of the original loan amount is permitted annually.

7. FIRST-TIME BUYER PROVISIONS
First-time buyers benefit from exemption from transfer tax (overdrachtsbelasting) on properties up to EUR 510,000.
The Starter Loan (Starterslening) may supplement the ING mortgage for eligible first-time buyers.
ING offers a dedicated first-time buyer advisory service at all branches.
'''

## Part 1 — Chunking Strategy Comparison

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Strategy A: Fixed-size chunking
def fixed_chunks(text, chunk_size=200, overlap=40):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

# Strategy B: Section-aware chunking (split on numbered sections)
def section_chunks(text):
    sections = re.split(r'(?=\n\d+\.\s+[A-Z])', text)
    return [s.strip() for s in sections if len(s.strip()) > 50]

# Strategy C: Sentence-aware chunking
def sentence_chunks(text, max_words=150, overlap_sentences=1):
    sentences = nltk.sent_tokenize(text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        w = len(sent.split())
        if current_len + w > max_words and current:
            chunks.append(' '.join(current))
            current = current[-overlap_sentences:]
            current_len = sum(len(s.split()) for s in current)
        current.append(sent)
        current_len += w
    if current:
        chunks.append(' '.join(current))
    return chunks

fixed   = fixed_chunks(ING_MORTGAGE_POLICY)
sections = section_chunks(ING_MORTGAGE_POLICY)
sentence = sentence_chunks(ING_MORTGAGE_POLICY)

print('Chunking Strategy Comparison:')
print(f'  Fixed-size ({len(fixed)} chunks, avg {sum(len(c.split()) for c in fixed)//len(fixed)} words each)')
print(f'  Section-aware ({len(sections)} chunks, avg {sum(len(c.split()) for c in sections)//len(sections)} words each)')
print(f'  Sentence-aware ({len(sentence)} chunks, avg {sum(len(c.split()) for c in sentence)//len(sentence)} words each)')

# Use section-aware for this lab — best for structured policy documents
chunks = sections
print(f'\nUsing section-aware chunks. First chunk preview:')
print(chunks[0][:200] + '...')

## Part 2 — ChromaDB Indexing and Dense Retrieval

In [ ]:
# Build ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name='ing_policy', get_or_create=True)

embeddings = embed_model.encode(chunks, normalize_embeddings=True)

collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=[{'section': i, 'word_count': len(c.split())} for i, c in enumerate(chunks)]
)

print(f'Indexed {collection.count()} chunks in ChromaDB')

# Test retrieval
def dense_search(query: str, k: int = 3) -> list:
    q_emb = embed_model.encode(query, normalize_embeddings=True)
    results = collection.query(query_embeddings=[q_emb.tolist()], n_results=k)
    return results['documents'][0]

test_query = 'What is the maximum mortgage for someone earning 80000 euros?'
results = dense_search(test_query)
print(f'\nQuery: "{test_query}"')
print(f'Top result (first 200 chars): {results[0][:200]}...')

## Part 3 — Hybrid Search with BM25 + Reciprocal Rank Fusion

In [ ]:
# BM25 sparse retrieval
tokenised_corpus = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenised_corpus)

def bm25_search(query: str, k: int = 5) -> list:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top_idx if scores[i] > 0]

def hybrid_rrf(query: str, k: int = 3, rrf_k: int = 60) -> list:
    # Dense results (ranked by cosine similarity)
    q_emb = embed_model.encode(query, normalize_embeddings=True)
    dense_results = collection.query(query_embeddings=[q_emb.tolist()], n_results=10)
    dense_docs = dense_results['documents'][0]

    # Sparse results (BM25)
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    sparse_top_idx = np.argsort(bm25_scores)[::-1][:10]
    sparse_docs = [chunks[i] for i in sparse_top_idx]

    # Reciprocal Rank Fusion
    scores = {}
    for rank, doc in enumerate(dense_docs):
        scores[doc[:50]] = scores.get(doc[:50], 0) + 1 / (rrf_k + rank + 1)
    for rank, doc in enumerate(sparse_docs):
        scores[doc[:50]] = scores.get(doc[:50], 0) + 1 / (rrf_k + rank + 1)

    # Merge and re-lookup full text
    all_docs = {doc[:50]: doc for doc in dense_docs + sparse_docs}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [all_docs[key] for key, _ in ranked]

# Comparison for a query with technical terminology
query = 'NHG guarantee maximum amount'
print(f'Query: "{query}"\n')
print('Dense only:', dense_search(query, k=1)[0][:150])
print()
print('BM25 only:', bm25_search(query, k=1)[0][:150] if bm25_search(query, k=1) else 'No keyword match')
print()
print('Hybrid RRF:', hybrid_rrf(query, k=1)[0][:150])

## Part 4 — Measuring Retrieval Quality with Hit Rate and MRR

In [ ]:
# Golden test set: (query, keyword that must appear in the top-retrieved chunk)
golden_set = [
    ('maximum mortgage income multiplier', 'gross annual income'),
    ('LTV for second property investment', '90%'),
    ('stress test interest rate', '5.0%'),
    ('required documents for mortgage application', 'werkgeversverklaring'),
    ('penalty free overpayment limit', '10%'),
    ('NHG national mortgage guarantee limit', '435,000'),
    ('first time buyer transfer tax exemption', 'overdrachtsbelasting'),
    ('leasehold property requirements', 'erfpacht'),
]

def evaluate_retrieval(search_fn, golden_set, k=3):
    hit_rate, mrr_scores = 0, []
    for query, expected_keyword in golden_set:
        results = search_fn(query, k=k)
        hit = False
        for rank, doc in enumerate(results, start=1):
            if expected_keyword.lower() in doc.lower():
                hit = True
                mrr_scores.append(1 / rank)
                break
        if not hit:
            mrr_scores.append(0.0)
        hit_rate += int(hit)
    return hit_rate / len(golden_set), sum(mrr_scores) / len(mrr_scores)

with mlflow.start_run(run_name='retrieval-comparison'):
    dense_hr, dense_mrr = evaluate_retrieval(dense_search, golden_set)
    hybrid_hr, hybrid_mrr = evaluate_retrieval(hybrid_rrf, golden_set)

    mlflow.log_metrics({'dense_hit_rate': dense_hr, 'dense_mrr': dense_mrr,
                        'hybrid_hit_rate': hybrid_hr, 'hybrid_mrr': hybrid_mrr})

    print('Retrieval Quality Comparison (k=3):')
    print(f'  Dense only  — Hit Rate: {dense_hr:.1%}  MRR: {dense_mrr:.3f}')
    print(f'  Hybrid RRF  — Hit Rate: {hybrid_hr:.1%}  MRR: {hybrid_mrr:.3f}')
    print('\nMetrics logged to MLflow experiment: ING-RAG-Pipeline')

## Part 5 — Full RAG Pipeline and RAGAS Evaluation

In [ ]:
def rag_answer(question: str, retrieval_fn=hybrid_rrf, k: int = 3) -> dict:
    retrieved = retrieval_fn(question, k=k)
    context = '\n\n'.join(retrieved)

    response = client_oai.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '''You are an ING mortgage advisor assistant.
Answer questions using ONLY the provided policy context.
If the context does not contain the answer, say: "I cannot find this information in the current policy document."
Be precise with numbers and percentages.'''},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}
        ],
        temperature=0,
        max_tokens=300,
    )
    return {
        'question': question,
        'answer': response.choices[0].message.content,
        'contexts': retrieved,
    }

# Test questions
test_questions = [
    'I earn 75,000 euros per year. What is the maximum mortgage I can get?',
    'What interest rate does ING use for the stress test?',
    'Can I get a mortgage on a property abroad?',
]

print('RAG Pipeline Test Results:')
print('=' * 60)
for q in test_questions:
    result = rag_answer(q)
    print(f'Q: {q}')
    print(f'A: {result["answer"][:200]}...')
    print()

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

eval_data = [
    {'question': 'What is the maximum LTV for a primary residence?',
     'ground_truth': 'The maximum LTV for a primary owner-occupied residence is 100% of market value.'},
    {'question': 'What monthly obligation limit applies in the stress test?',
     'ground_truth': 'Total monthly obligations may not exceed 35% of gross monthly income.'},
    {'question': 'How much can I overpay annually without penalty?',
     'ground_truth': 'You can overpay up to 10% of the original loan amount per year without penalty.'},
]

# Get RAG answers for eval
for item in eval_data:
    result = rag_answer(item['question'])
    item['answer'] = result['answer']
    item['contexts'] = result['contexts']

dataset = Dataset.from_list(eval_data)

with mlflow.start_run(run_name='ragas-evaluation'):
    scores = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    scores_df = scores.to_pandas()
    mean_scores = scores_df.mean(numeric_only=True).to_dict()

    mlflow.log_metrics({k: v for k, v in mean_scores.items() if isinstance(v, float)})

    print('RAGAS Evaluation Results:')
    for metric, score in mean_scores.items():
        if isinstance(score, float):
            status = 'PASS' if score >= 0.75 else 'REVIEW'
            print(f'  {metric:<35} {score:.3f}  [{status}]')
    print('\nMetrics logged to MLflow.')

## ✅ Lab 04 Complete

You have built a production-style RAG system:
- **3 chunking strategies** compared — section-aware won for structured policy documents
- **ChromaDB vector store** — indexed, queried, metadata-filtered
- **Hybrid search (BM25 + dense + RRF)** — measurably better Hit Rate and MRR
- **Full RAG pipeline** — context assembly, grounded generation, out-of-scope handling
- **RAGAS evaluation** — faithfulness, answer relevancy, context precision tracked in MLflow

**Next:** Lab 05 — AI Agents